In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

### Preparation

In [25]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [26]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1. How many lesson pages

In [27]:
len(documents)

72

### Q2. Indexing and searching

In [28]:
documents[0].keys()


dict_keys(['content', 'filename'])

In [29]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [40]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    # boost_dict={'content': 2.0, '...': 0.5},
    # filter_dict={'filename': '...'},
    num_results=5
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

### Q3. RAG

In [97]:
import importlib
import rag_helper

importlib.reload(rag_helper)

from rag_helper import RAGBase

In [98]:

from rag_helper import RAGBase

rag_base = RAGBase(
    index=index,
    llm_client=openai_client,
    #filename="01-agentic-rag/lessons/14-agentic-loop.md",
    model="gpt-5.4-mini"
)

rag_base



In [99]:
question = "How does the agentic loop keep calling the model until it stops?"

answer = rag_base.rag(question)
answer

'It keeps looping with a `while True` loop and checks whether the model made any `function_call`s on that turn.\n\n- If there was at least one function call, the code runs the tool, appends the result to `messages`, and continues.\n- If there were no function calls, that means the model returned a final answer, so the loop breaks.\n\nIn the code, this is the exit condition:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the agentic loop stops when the model stops asking for tools.'

In [100]:
llm_response = rag_base.get_llm_response()
llm_response.usage.input_tokens

2231

### Q4. Chunking

In [101]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [102]:
len(chunks)

295

### Q5. RAG with chunking

In [103]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

In [104]:
from rag_helper import RAGBase

rag_base_chunked = RAGBase(index=index, llm_client=openai_client, model="gpt-5.4-mini")
rag_base_chunked

In [105]:
question = "How does the agentic loop keep calling the model until it stops?"
rag_base_chunked.rag(question)

llm_response2 = rag_base_chunked.get_llm_response()
llm_response2.usage.input_tokens

2231

In [107]:
llm_response.usage.input_tokens / llm_response2.usage.input_tokens

1.0

### Q6. Turning it into an agent

In [108]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [109]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lessons database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [110]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [111]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [112]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [113]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [114]:
question = "How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


### 